# Lab 08 · GPU offload · OpenMP `target` on Polaris

This lab moves from Crux (CPU-only) to **Polaris**, an ALCF supercomputer with NVIDIA A100 GPUs. Same physics, same C source, plus one directive: `#pragma omp target teams distribute parallel for` puts the loop on the GPU. This is the gentlest introduction to GPU programming; lab 09 rewrites the same kernel in CUDA to show what's really happening.

**Prerequisites.** Lab 03 (OpenMP), Lab 04 (OpenMP pitfalls including first-touch). Familiar with Polaris login (labMEM from a future lab, or ask on the class Slack). Change `host="polaris"` in setupLab and update your ~/.ssh/config accordingly.

**Builds toward.** Lab 09 (CUDA — writing the kernel by hand). Lab 10 (MPI+CUDA — multi-GPU).

> **📚 Where to look when you're stuck**
>
> - [**OpenMP target directive**](https://www.openmp.org/spec-html/5.2/openmpsu60.html)
> - [**Polaris system overview**](https://docs.alcf.anl.gov/polaris/)
> - [**Polaris compilers for GPU**](https://docs.alcf.anl.gov/polaris/compiling-and-linking/)



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]**, **[Hub -> Crux]**, **[Crux compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab08", host="polaris",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab08 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab08 dir ready')


## Part 1 · The `#pragma omp target` directive

OpenMP 4.5+ added the `target` construct for offloading compute to a device. The simplest form:

```c
#pragma omp target teams distribute parallel for collapse(2)
for (int i = 1; i < N-1; i++)
    for (int j = 1; j < N-1; j++)
        unew[i*N+j] = /* stencil */;
```

The compiler generates GPU code for the loop body. Data movement is either **implicit** (compiler figures out) or **explicit** (you write `map(to:...)` and `map(from:...)` clauses to control it). For a real code that runs many steps, you'll want explicit maps so the array lives on the GPU across steps.


In [ ]:
# [Hub -> Polaris] Take lab03's omp source, add target pragma + data region.
sshGet(env['HPC_LAB_DIR'].replace('lab08','lab03') + '/heat2Domp.c',
       str(labDir/'heat2Dgpu.c')) if pathlib.Path(labDir/'heat2Dgpu.c').exists() is False else None
print('Edit heat2Dgpu.c: replace `#pragma omp parallel for` with')
print('  `#pragma omp target teams distribute parallel for collapse(2)`')
print('and wrap the timestep loop in a `#pragma omp target data map(tofrom: u[:N*N])` block.')


In [ ]:
checkpoint("Part 1 - GPU source draft", [
    check("gpu source drafted", dirExists(str(labDir))),
])


## Part 2 · Build for Polaris GPU

Polaris uses NVIDIA HPC SDK compilers (or PrgEnv-nvhpc). The offload flag is:

```bash
cc -mp=gpu -gpu=cc80 -O3 -o heat2Dgpu heat2Dgpu.c
```

- `-mp=gpu` — enable OpenMP GPU offload
- `-gpu=cc80` — target compute capability 8.0 (A100)


In [ ]:
# [Hub -> Polaris] Build and run one GPU rank on one Polaris node.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
module load PrgEnv-nvhpc 2>/dev/null || module load nvhpc 2>/dev/null || true
cc -mp=gpu -gpu=cc80 -O3 -o heat2Dgpu heat2Dgpu.c -lm
./heat2Dgpu --N 2048 --steps 500 --snapEvery 0 --outDir ./out --variant gpu-omp
cat ./out/timings.csv || true
'''
pbsPath = labDir/'gpuJob.pbs'
# Polaris debug queue: one GPU node
pbsPath.write_text(pbsHeader(name='lab08GPU', project=env['HPC_PROJECT'],
                             queue='debug', select='1:system=polaris', walltime='00:15:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/gpu.out') + jobBody)
sshPut(str(labDir/'heat2Dgpu.c'), env['HPC_LAB_DIR']+'/heat2Dgpu.c')
sshPut(str(pbsPath),             env['HPC_LAB_DIR']+'/gpuJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/gpuJob.pbs'); waitJob(jobID, 30, 1200)
sshGet(env['HPC_LAB_DIR']+'/gpu.out', str(labDir/'gpu.out'))
print((labDir/'gpu.out').read_text())


In [ ]:
checkpoint("Part 2 - GPU build and run", [
    check("gpu output", fileExists(str(labDir/'gpu.out'))),
])


## Part 3 · Compare to CPU

Pull lab 06's or lab 07's best CPU MLUP/s and compare to the GPU number. Expect a 5-20x speedup for a well-tuned OMP target kernel; the exact number depends on how much of the time is now dominated by host→device transfers.


In [ ]:
# [Hub] Fetch previous best CPU numbers if present and compare.
print('Compare gpu.out mlups to lab 07 hyb.out best mlups')
print('Ratio > 5x is a healthy OMP-target result.')
print('Ratio > 20x usually means the CPU comparison was untuned.')


In [ ]:
checkpoint("Part 3 - compared to CPU", [
    check("gpu.out present", fileExists(str(labDir/'gpu.out'))),
])


## Part 4 · Data movement is the enemy

The single biggest performance mistake in GPU programming is copying data between host and device more than necessary. Every `map(tofrom:...)` on the outermost loop = one round-trip per step. **Wrap the whole timestep loop in a `#pragma omp target data` region so the array lives on the GPU.**

Verify by running with and without the outer data region and comparing wall.


In [ ]:
# [Hub -> Polaris] Optional: separate build without the target data wrapper.
showNote('Try commenting the outer `#pragma omp target data map(tofrom: u)` region\n'
         'and re-running. You should see 5-10x slowdown - that is the transfer cost.',
         kind='info')


In [ ]:
checkpoint("Part 4 - understood data movement", [
    check("lab dir persistent", dirExists(str(labDir))),
])


## Part 5 · Correctness

The GPU version and the CPU version should agree on `sumU` to ~1e-8 relative (GPU FMAs can differ slightly from the CPU; labCC Part 6 warned about this). If they disagree by more than that, you probably have a stale halo or a boundary bug.


In [ ]:
checkpoint("Part 5 - correctness note", [
    check("gpu.out present", fileExists(str(labDir/'gpu.out'))),
])


## Part 6 · Bridge to lab 09

OpenMP `target` is one directive. **CUDA** is the same computation written by hand: you write the kernel, you launch it with `<<<grid, block>>>`, you manage `cudaMemcpy` yourself. The upside: you learn what the compiler was doing for you, and you can hand-tune what OpenMP wouldn't.


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("GPU offload")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("GPU offload")
